This is a short notebook to retrieve datasets from NFDI4Earth fulfilling specific conditions. The data retrieved can be stored as json afterwards.
The following metadata are currently considered (Corresponding query: *access_desired_attributes_only*)


*   Dataset title
*   Dataset description
*   Dataset url (a.ka. landing page)
*   Dataset download url
*   Temporal coverage (start date and end date)
*   Location





In [20]:
!pip install sparqlwrapper

# Query datasets

## Specify endpoint and formulate queries

In [21]:
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint = SPARQLWrapper("https://sparql.knowledgehub.nfdi4earth.de/")
endpoint.setReturnFormat(JSON)


access_available_attributes_for_datasets = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>

SELECT DISTINCT ?predicate
WHERE {
  ?entity rdf:type dcat:Dataset;
          ?predicate ?attributeInformation.
}

"""

access_datasets_and_all_attributes = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>

SELECT *
WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             #dcat:landingPage ?landingpage;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?startDate;
                 dcat:endDate ?endDate.
   ?bnode_spatial locn:geometry ?location.
}

"""


access_dataset_title = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>

SELECT DISTINCT ?title

WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             #dcat:landingPage ?landingpage;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?startDate;
                 dcat:endDate ?endDate.
   ?bnode_spatial locn:geometry ?location.
}

"""


access_desired_attributes_only = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>

SELECT DISTINCT ?title (GROUP_CONCAT(DISTINCT ?authorName; separator=", ") AS ?authors) ?description ?landingpage ?download_url ?startDate ?endDate ?location

WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             dcat:landingPage ?landingpage;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?startDate;
                 dcat:endDate ?endDate.
   ?bnode_spatial locn:geometry ?location.
}

"""

access_desired_attributes_29082026 = """

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX schema: <http://schema.org/>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX locn: <http://www.w3.org/ns/locn#>
PREFIX n4e: <http://nfdi4earth.de/ontology/>

SELECT DISTINCT ?title (GROUP_CONCAT(DISTINCT ?authorName; separator=", ") AS ?authors) ?description ?url ?download_url ?temporalCoverageStartDate ?temporalCoverageEndDate ?location ?release_date

WHERE {
  ?dataset rdf:type dcat:Dataset;
             dct:title ?title;
             schema:description ?description;
             dcat:landingPage ?url;
             dct:creator ?bnode_creator;
             dcat:distribution ?bnode_distrib;
             dct:temporal ?bnode_temporal;
             dct:spatial ?bnode_spatial.

   ?bnode_creator schema:name ?autorname.
   ?bnode_distrib dcat:downloadURL ?download_url.

   ?bnode_temporal dcat:startDate ?temporalCoverageStartDate;
                 dcat:endDate ?temporalCoverageEndDate.
   ?bnode_spatial locn:geometry ?location.

   OPTIONAL {?dataset dct:issued ?release_date.}

}

"""

## Execute queries




In [22]:
import json
import time

datasets = []
n_iterations = 10 # virtuoso gives back 10000 rows max at once. If you want to retrieve more than that, you need to iterate, e.g. you need 2 iterations if you want to retrieve 18000 items, or 50 iterations for 500 000 items and so on.

try:

    for i in range(n_iterations):

      # pick the query to execute
      endpoint.setQuery(access_desired_attributes_29082026 + " OFFSET " + str(i) + "0000" + " LIMIT 10000") # since virtuoso returns 10000 rows max, we pick 10000 max per iteration.

      response = endpoint.queryAndConvert()

      all_results = response["results"]["bindings"]

      print("Iteration: ",  str(i+1),  "; the number of results is ", len(all_results))

      for result in all_results:

        datasets.append(result)


    # format the results as json strings
    results_as_json_str = json.dumps(datasets)

    timestr = time.strftime("%Y%m%d-%H%M%S")
    json_path = "sample_" + timestr + ".json"

    # optional: save all results in a file
    with open(json_path, "w") as f:
      f.write(results_as_json_str)

except Exception as e:
    print(e)

Iteration:  1 ; the number of results is  10000
Iteration:  2 ; the number of results is  10000
Iteration:  3 ; the number of results is  10000
Iteration:  4 ; the number of results is  10000
Iteration:  5 ; the number of results is  10000
Iteration:  6 ; the number of results is  10000
Iteration:  7 ; the number of results is  8461
Iteration:  8 ; the number of results is  0
Iteration:  9 ; the number of results is  0
Iteration:  10 ; the number of results is  0


## Display results

In [23]:
print ("The number of results is: ", len(datasets))

The number of results is:  68461


Read the results and display the first 10

In [24]:
data = json.loads(results_as_json_str)

for i, datum in enumerate(data):
    if i < 10:
      print(datum)

{'title': {'type': 'literal', 'value': 'Modelldaten Ertragspotenzial in verschiedenen Ausbaujahren Offshore-Windenergie Nordsee -2030-2006-10'}, 'authors': {'type': 'literal', 'value': ''}, 'description': {'type': 'literal', 'value': 'Die Daten umfassen die im deutschen Hoheitsgebiet und der ausschließlichen Wirtschaftszone (AWZ) errechneten Erträge von Windparks und die dafür relevanten Windbedingungen unter der Annahme eines Ausbaus der Offshore Windenergie in kontinuierlichen Ausbaujahren. Die Berechnungen wurden mit dem numerischen Wettermodell WRF unter Nutzung einer Parametrisierung von Windparks nach Fitch durchgeführt. Die Daten liegen in 10-minütlicher zeitlicher und 2 km x 2 km räumlicher Auflösung für die Nordsee für das meteorologische Jahr 2006 vor. Die Jahreszahlen der Dateien beziehen sich auf das jeweilige Ausbaujahr. Die Variablen des Datensatzes sind Windgeschwindigkeit (WS) und Windrichtung (WD) auf 9 Höhenleveln zwischen 50 m und 350 m, die Leistung (POWER) der Wind

Store results in a file (takes time for large files, so commented by default)

In [25]:
'''
import json

with open(json_path, "r") as file:
    data = json.load(file)

print(data)
'''

'\nimport json\n\nwith open(json_path, "r") as file:\n    data = json.load(file)\n\nprint(data)\n'